# Visualisation 2 — Regional Name Effects in France

**Research questions:**
- Is there a regional effect in naming data?
- Are some names more popular in specific departments?
- Are nationally popular names uniformly popular across France?

**Dashboard:**
- **Map 1** — Each department coloured by η: how much the local #1 name beats the national #1 name locally. High η = strong regional naming identity.
- **Map 2** — Click any department on Map 1 to compare its naming culture to every other department (η_k,S,R metric, k=3 top names).
- **Decade dropdown** — Re-renders both maps for the selected decade.

> The GeoJSON is fetched by the browser from GitHub at render time. An internet connection is required.

## 1 — Imports

In [ ]:
import json
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# GeoJSON is served from GitHub so the browser can fetch it without embedding
# the full geometry inline (which would produce a 15+ MB Vega-Lite spec).
GEO_URL = (
    'https://raw.githubusercontent.com/gregoiredavid/'
    'france-geojson/master/departements-avec-outre-mer.geojson'
)

## 2 — Load and filter data

In [ ]:
df = pd.read_csv(
    '../dpt2020.csv',
    sep=';',
    dtype={'dpt': str, 'annais': str, 'sexe': int, 'nombre': int},
)

# Remove national aggregates (XX), year aggregates (XXXX), and rare-name bucket
df = df[
    (df['dpt'] != 'XX') &
    (df['annais'] != 'XXXX') &
    (df['preusuel'] != '_PRENOMS_RARES')
].copy()

df['annais'] = df['annais'].astype(int)
df['decade'] = (df['annais'] // 10) * 10

print(f"Rows after filter: {len(df):,}")
print(f"Departments: {df['dpt'].nunique()}")
print(f"Decades: {sorted(df['decade'].unique())}")

## 3 — Aggregate by (decade, dpt, preusuel)

In [ ]:
agg = (
    df.groupby(['decade', 'dpt', 'preusuel'], sort=False)['nombre']
    .sum()
    .reset_index()
)

agg_gender = (
    df.groupby(['decade', 'dpt', 'preusuel', 'sexe'], sort=False)['nombre']
    .sum()
    .reset_index()
)

print(f"Aggregated rows: {len(agg):,}")

## 4 — Compute η per (decade, dpt) for Map 1

$$\eta_{dpt} = \frac{|\text{local top name}|_{dpt}}{|\text{national top name}|_{dpt}} \geq 1$$

η = 1: the local #1 name IS the national #1 name.  
η > 1: a different name is locally more popular than the national #1.

In [ ]:
K_DISPLAY  = 5  # names shown in Map 1 hover tooltip
K_PAIRWISE = 3  # names used in Map 2 pairwise η

eta_rows = []
for decade, d in agg.groupby('decade'):
    global_top = d.groupby('preusuel')['nombre'].sum().idxmax()

    local_top = (
        d.sort_values('nombre', ascending=False)
        .groupby('dpt', sort=False)
        .first()
        .reset_index()
        .rename(columns={'preusuel': 'top_name', 'nombre': 'top_count'})
    )

    global_in_dept = (
        d[d['preusuel'] == global_top][['dpt', 'nombre']]
        .rename(columns={'nombre': 'global_count'})
    )

    merged = local_top.merge(global_in_dept, on='dpt', how='left')
    merged['global_count'] = merged['global_count'].fillna(1)
    merged['eta'] = merged['top_count'] / merged['global_count']
    merged['decade'] = decade
    eta_rows.append(merged[['decade', 'dpt', 'eta', 'top_name']])

eta_long = pd.concat(eta_rows, ignore_index=True)
print(eta_long.shape)
eta_long.head()

## 5 — Format top-k tooltip string per (decade, dpt)

In [ ]:
gender_wide = agg_gender.pivot_table(
    index=['decade', 'dpt', 'preusuel'],
    columns='sexe',
    values='nombre',
    fill_value=0,
).reset_index()
gender_wide.columns.name = None

for col in [1, 2]:
    if col not in gender_wide.columns:
        gender_wide[col] = 0
gender_wide = gender_wide.rename(columns={1: 'count_m', 2: 'count_f'})
gender_wide['total'] = gender_wide['count_m'] + gender_wide['count_f']
gender_wide['pct_m'] = (
    (gender_wide['count_m'] / gender_wide['total'].replace(0, np.nan)) * 100
).round(0).fillna(0).astype(int)

agg_g = agg.merge(
    gender_wide[['decade', 'dpt', 'preusuel', 'pct_m']],
    on=['decade', 'dpt', 'preusuel'],
    how='left',
)

def fmt_topk(group: pd.DataFrame) -> str:
    top = group.nlargest(K_DISPLAY, 'nombre')
    return ' | '.join(
        f"{r['preusuel']} ({int(r['nombre']):,}, {int(r['pct_m'])}%M)"
        for _, r in top.iterrows()
    )

topk_str = (
    agg_g.groupby(['decade', 'dpt'], sort=False)
    .apply(fmt_topk, include_groups=False)
    .reset_index(name='top_names_str')
)

eta_long = eta_long.merge(topk_str, on=['decade', 'dpt'], how='left')
eta_long.head(3)

## 6 — Compute pairwise η_k,S,R for Map 2

$$\eta_{k,S,R} = \frac{\sum_{i=1}^{k}|\text{i-th top name of } R|_R}{\sum_{i=1}^{k}|\text{i-th top name of } S|_R} \geq 1$$

η_k,S,R ≈ 1 → R has a similar naming culture to S.  
η_k,S,R >> 1 → R's own names are much more popular there than S's top names → very different cultures.

*This cell takes ~30–60 s for 12 decades × 96 departments.*

In [ ]:
pairwise_rows = []

for decade, d in agg.groupby('decade'):
    topk = (
        d.sort_values('nombre', ascending=False)
        .groupby('dpt', sort=False)
        .head(K_PAIRWISE)
    )
    topk_sums = topk.groupby('dpt')['nombre'].sum()

    pivot = d.pivot_table(
        index='preusuel',
        columns='dpt',
        values='nombre',
        fill_value=0,
        aggfunc='sum',
    )

    for S, s_group in topk.groupby('dpt'):
        s_names = s_group['preusuel'].tolist()
        s_in_r = pivot.loc[pivot.index.isin(s_names)].sum()
        eta_series = topk_sums / s_in_r.replace(0, np.nan)

        for R, val in eta_series.items():
            pairwise_rows.append({
                'decade': decade,
                'dpt':    S,
                'R_dpt':  R,
                'eta_kSR': val,
            })

pairwise_long = pd.DataFrame(pairwise_rows)
print(f"Pairwise rows: {len(pairwise_long):,}")
pairwise_long.head()

## 7 — Cache GeoJSON locally and build department name lookup

In [ ]:
geo_path = Path('departements-avec-outre-mer.geojson')

if not geo_path.exists():
    print('Downloading GeoJSON …')
    r = requests.get(GEO_URL, timeout=30)
    r.raise_for_status()
    geo_path.write_bytes(r.content)
    print('Saved.')
else:
    print('GeoJSON already cached.')

with open(geo_path, encoding='utf-8') as f:
    geo_json = json.load(f)

# Lightweight lookup table — no geometry column, so no massive inline JSON
dept_names = pd.DataFrame([
    {'code': feat['properties']['code'], 'nom': feat['properties']['nom']}
    for feat in geo_json['features']
])

print(f"Departments in GeoJSON: {len(dept_names)}")
dept_names.head()

## 8 — Interactive Altair dashboard

**How to use:**
1. Choose a decade from the dropdown — both maps update.
2. Hover over any department on **Map 1** to see η and the top-5 names.
3. Click a department on **Map 1** → **Map 2** shows how culturally similar every other department is.

**Why ipywidgets for the decade selector?**  
Pre-filtering in Python gives Altair one row per department (instead of one row per decade × department). This avoids the multi-row lookup ambiguity in Vega-Lite's `transform_lookup` and keeps the Vega-Lite spec small.

In [ ]:
from ipywidgets import interact, widgets


def make_dashboard(decade: int):
    # ── Pre-filter to the selected decade in Python ───────────────────────────
    eta_d = (
        eta_long[eta_long['decade'] == decade]
        [['dpt', 'eta', 'top_name', 'top_names_str']]
        .reset_index(drop=True)
    )

    # Wide pairwise table: one row per R_dpt, one column per S dept code.
    # Column names ARE the dept codes (e.g. '75', '2A').
    # After Altair fold(as_=['dpt', 'eta_kSR']), the 'dpt' column holds those
    # same dept codes — identical to what dept_sel captures on Map 1 click —
    # so transform_filter(dept_sel) works correctly across the hconcat.
    pair_wide = (
        pairwise_long[pairwise_long['decade'] == decade]
        [['R_dpt', 'dpt', 'eta_kSR']]
        .pivot(index='R_dpt', columns='dpt', values='eta_kSR')
        .replace([np.inf, -np.inf], 10.0)
        .fillna(1.0)
        .clip(upper=10.0)
        .reset_index()
    )
    pair_wide.columns.name = None
    dept_code_cols = [c for c in pair_wide.columns if c != 'R_dpt']

    geo = alt.UrlData(
        url=GEO_URL,
        format=alt.DataFormat(property='features', type='json'),
    )

    # Exclude overseas departments (971–976): Guyane is in South America and
    # would compress metropolitan France to a tiny sliver in Mercator projection.
    metro = "datum.properties.code < '97'"

    dept_sel = alt.selection_point(fields=['dpt'], name='dept_sel', empty='none')

    # ── Map 1: η choropleth ───────────────────────────────────────────────────
    map1 = (
        alt.Chart(geo)
        .mark_geoshape(stroke='white', strokeWidth=0.5)
        .transform_filter(metro)
        .transform_calculate(
            dpt='datum.properties.code',
            nom='datum.properties.nom',
        )
        .transform_lookup(
            lookup='dpt',
            from_=alt.LookupData(eta_d, 'dpt', ['eta', 'top_name', 'top_names_str']),
        )
        .encode(
            color=alt.condition(
                dept_sel,
                alt.Color(
                    'eta:Q',
                    scale=alt.Scale(scheme='yelloworangered', domainMin=1.0),
                    title='η',
                    legend=alt.Legend(orient='bottom', gradientLength=180),
                ),
                alt.value('#a8d8f0'),
            ),
            tooltip=[
                alt.Tooltip('nom:N',           title='Department'),
                alt.Tooltip('eta:Q',           title='η',          format='.3f'),
                alt.Tooltip('top_name:N',      title='Local #1'),
                alt.Tooltip('top_names_str:N', title=f'Top {K_DISPLAY} names'),
            ],
        )
        .add_params(dept_sel)
        .project('mercator')
        .properties(
            width=480, height=520,
            title=alt.TitleParams(
                text='η — Regional Name Distinctiveness',
                subtitle=f'{decade}s  |  Click a department to compare →',
            ),
        )
    )

    # ── Map 2: η_k,S,R comparison ─────────────────────────────────────────────
    # GeoJSON is primary data for both layers so shapes render natively.
    #
    # map2_fg pipeline:
    #   1. filter metro France
    #   2. calculate R_dpt from properties.code
    #   3. lookup pair_wide by R_dpt  → adds one column per S dept (dept_code_cols)
    #   4. fold dept_code_cols as ['dpt', 'eta_kSR']
    #      → 'dpt' now holds S dept codes, same values dept_sel captures on click
    #   5. transform_filter(dept_sel) keeps only dpt == clicked dept (one row per feature)
    map2_bg = (
        alt.Chart(geo)
        .mark_geoshape(fill='#e0e0e0', stroke='white', strokeWidth=0.5)
        .transform_filter(metro)
        .project('mercator')
        .properties(width=480, height=520)
    )

    map2_fg = (
        alt.Chart(geo)
        .mark_geoshape(stroke='white', strokeWidth=0.5)
        .transform_filter(metro)
        .transform_calculate(
            R_dpt='datum.properties.code',
            nom='datum.properties.nom',
        )
        .transform_lookup(
            lookup='R_dpt',
            from_=alt.LookupData(pair_wide, 'R_dpt', dept_code_cols),
        )
        .transform_fold(dept_code_cols, as_=['dpt', 'eta_kSR'])
        .transform_filter(dept_sel)
        .encode(
            color=alt.Color(
                'eta_kSR:Q',
                scale=alt.Scale(scheme='redyellowgreen', reverse=True, domainMin=1.0),
                title='η_k,S,R',
                legend=alt.Legend(orient='bottom', gradientLength=180),
            ),
            tooltip=[
                alt.Tooltip('nom:N',     title='Department'),
                alt.Tooltip('eta_kSR:Q', title='η_k,S,R', format='.3f'),
            ],
        )
        .project('mercator')
        .properties(width=480, height=520)
    )

    map2 = alt.layer(map2_bg, map2_fg).properties(
        title=alt.TitleParams(
            text='η_k,S,R — Naming Culture Similarity',
            subtitle=f'Green ≈ similar to selected dept  |  k = {K_PAIRWISE} names',
        )
    )

    return (
        alt.hconcat(map1, map2)
        .resolve_scale(color='independent')
    )


decades_list = sorted(eta_long['decade'].unique().tolist())
interact(
    make_dashboard,
    decade=widgets.Dropdown(
        options=[(f'{d}s', d) for d in decades_list],
        value=2000,
        description='Decade:',
        style={'description_width': 'initial'},
    ),
)